In [1]:

import pandas as pd
import requests
import time
import os
from dotenv import load_dotenv
from urllib.parse import quote

class NaverSearchAPI:
    def __init__(self):
        # .env 파일에서 환경변수 로드
        load_dotenv()
        
        self.client_id = os.getenv('Client_ID')
        self.client_secret = os.getenv('Client_Secret')
        
        # API 키 확인
        if not self.client_id or not self.client_secret:
            raise ValueError("API 키가 설정되지 않았습니다. .env 파일을 확인하세요.")
    
    def get_search_count(self, keyword, search_type="blog"):
        """네이버 검색 API로 검색 결과 수 가져오기"""
        
        # API URL 설정
        api_urls = {
            "blog": "https://openapi.naver.com/v1/search/blog.json",
            "news": "https://openapi.naver.com/v1/search/news.json",
            "cafe": "https://openapi.naver.com/v1/search/cafearticle.json",
            "web": "https://openapi.naver.com/v1/search/webkr.json"
        }
        
        url = api_urls.get(search_type, api_urls["blog"])
        
        # 헤더 설정
        headers = {
            'X-Naver-Client-Id': self.client_id,
            'X-Naver-Client-Secret': self.client_secret
        }
        
        # 파라미터 설정
        params = {
            'query': keyword,
            'display': 1,  # 1개만 가져와서 total만 확인
            'start': 1
        }
        
        try:
            response = requests.get(url, headers=headers, params=params)
            
            if response.status_code == 200:
                data = response.json()
                return data.get('total', 0)
            else:
                print(f"API 오류 ({search_type}): {response.status_code} - {keyword}")
                return 0
                
        except Exception as e:
            print(f"API 호출 오류: {e} - {keyword}")
            return 0
    
    def test_api_connection(self):
        """API 연결 테스트"""
        test_keyword = "음식 김치찌개"
        test_result = self.get_search_count(test_keyword)
        
        if test_result > 0:
            print(f"API 연결 테스트 성공: '{test_keyword}' 블로그 검색 결과 {test_result:,}개")
            return True
        else:
            print("API 연결 테스트 실패. .env 파일의 API 키를 확인하세요.")
            return False
    
    def collect_search_counts(self, keywords):
        """모든 키워드의 검색 결과 수 수집 ('음식' 접두사 포함)"""
        
        results = []
        total_keywords = len(keywords)
        
        print(f"총 {total_keywords}개 키워드 검색 시작 (모든 키워드 앞에 '음식' 추가)")
        print("-" * 50)
        
        for i, keyword in enumerate(keywords, 1):
            # 키워드 앞에 '음식' 붙이기
            search_keyword = f"음식 {keyword}"
            print(f"[{i}/{total_keywords}] {keyword} -> {search_keyword}")
            
            # 각 검색 타입별로 결과 수 가져오기
            blog_count = self.get_search_count(search_keyword, "blog")
            time.sleep(0.1)
            
            news_count = self.get_search_count(search_keyword, "news")
            time.sleep(0.1)
            
            cafe_count = self.get_search_count(search_keyword, "cafe")
            time.sleep(0.1)
            
            web_count = self.get_search_count(search_keyword, "web")
            time.sleep(0.1)
            
            # 결과 저장
            total_count = blog_count + news_count + cafe_count + web_count
            
            results.append({
                '원본키워드': keyword,
                '검색키워드': search_keyword,
                '블로그_검색수': blog_count,
                '뉴스_검색수': news_count,
                '카페_검색수': cafe_count,
                '웹_검색수': web_count,
                '총합': total_count
            })
            
            print(f"  블로그: {blog_count:,}, 뉴스: {news_count:,}, 카페: {cafe_count:,}, 웹: {web_count:,}")
            print(f"  총합: {total_count:,}")
            
            # API 호출 제한을 위한 대기
            time.sleep(0.5)
        
        return pd.DataFrame(results)

def load_keywords_from_csv():
    """CSV 파일에서 키워드 읽기"""
    try:
        print("CSV 파일 읽기 중...")
        df = pd.read_csv('식당대12중53소132상세메뉴379분류.csv')
        
        keywords = []
        for menu in df['상세메뉴'].dropna():
            for item in str(menu).split(','):
                item = item.strip()
                if item:
                    keywords.append(item)
        
        print(f"총 {len(keywords)}개 키워드 로딩 완료")
        return keywords
        
    except FileNotFoundError:
        print("오류: '식당대12중53소132상세메뉴379분류.csv' 파일을 찾을 수 없습니다.")
        print("현재 폴더에 CSV 파일이 있는지 확인하세요.")
        return None
    except Exception as e:
        print(f"CSV 파일 읽기 오류: {e}")
        return None

def process_all_keywords(keywords):
    """전체 키워드 처리"""
    print(f"\n전체 {len(keywords)}개 키워드를 처리합니다.")
    return keywords

def save_results_to_excel(results_df):
    """결과를 엑셀 파일로 저장"""
    # 결과 정렬 (총합 기준 내림차순)
    results_df = results_df.sort_values('총합', ascending=False)
    
    # 엑셀 파일로 저장
    output_filename = "음식_키워드_검색량.xlsx"
    results_df.to_excel(output_filename, index=False)
    
    print("\n" + "=" * 50)
    print("수집 완료!")
    print(f"파일 저장: {output_filename}")
    print(f"총 처리된 키워드: {len(results_df)}개")
    
    # 상위 10개 키워드 출력
    print("\n상위 10개 키워드:")
    print("-" * 50)
    top_10 = results_df.head(10)
    for _, row in top_10.iterrows():
        print(f"{row['검색키워드']}: {row['총합']:,}개")
    
    # 통계 정보
    print(f"\n통계 정보:")
    print(f"평균 검색 결과 수: {results_df['총합'].mean():,.0f}개")
    print(f"최대 검색 결과 수: {results_df['총합'].max():,}개")
    print(f"최소 검색 결과 수: {results_df['총합'].min():,}개")
    
    return results_df

def create_env_file_template():
    """환경변수 파일 템플릿 생성"""
    env_template = """# 네이버 개발자센터에서 발급받은 API 키를 입력하세요
# https://developers.naver.com/apps/#/register

Client_ID=your_client_id_here
Client_Secret=your_client_secret_here
"""
    
    with open('.env', 'w', encoding='utf-8') as f:
        f.write(env_template)
    
    print(".env 파일 템플릿을 생성했습니다.")
    print("파일을 열어서 API 키를 입력한 후 다시 실행하세요.")

def main():
    print("네이버 검색 API를 사용한 음식 키워드 검색량 조사")
    print("모든 키워드 앞에 '음식'을 붙여서 검색합니다")
    print("=" * 60)
    
    # .env 파일 확인
    if not os.path.exists('.env'):
        print("오류: .env 파일이 없습니다.")
        print("현재 폴더에 .env 파일이 있는지 확인하세요.")
        return
    
    # API 클래스 초기화
    try:
        api = NaverSearchAPI()
    except ValueError as e:
        print(f"오류: {e}")
        print("\n.env 파일을 확인하세요:")
        print("Client_ID=your_client_id")
        print("Client_Secret=your_client_secret")
        return
    
    # API 연결 테스트
    print("\nAPI 연결 테스트 중...")
    if not api.test_api_connection():
        return
    
    # 키워드 로드
    keywords = load_keywords_from_csv()
    if keywords is None:
        return
    
    # 전체 키워드 처리
    selected_keywords = process_all_keywords(keywords)
    
    # 예상 소요 시간 계산
    estimated_time = len(selected_keywords) * 0.5 / 60  # 키워드당 0.5초 * 분 변환
    print(f"예상 소요 시간: 약 {estimated_time:.1f}분")
    
    # 처리 시작 확인
    start_confirm = input("\n처리를 시작하시겠습니까? (y/n): ").strip().lower()
    if start_confirm != 'y':
        print("처리를 중단합니다.")
        return
    
    # 검색 결과 수 수집
    try:
        print(f"\n전체 키워드 검색량 조사 시작...")
        results_df = api.collect_search_counts(selected_keywords)
        
        # 결과 저장 및 출력
        save_results_to_excel(results_df)
        
    except Exception as e:
        print(f"처리 중 오류 발생: {e}")
        return

if __name__ == "__main__":
    main()

네이버 검색 API를 사용한 음식 키워드 검색량 조사
모든 키워드 앞에 '음식'을 붙여서 검색합니다

API 연결 테스트 중...
API 연결 테스트 성공: '음식 김치찌개' 블로그 검색 결과 1,251,767개
CSV 파일 읽기 중...
총 381개 키워드 로딩 완료

전체 381개 키워드를 처리합니다.
예상 소요 시간: 약 3.2분

처리를 시작하시겠습니까? (y/n): y

전체 키워드 검색량 조사 시작...
총 381개 키워드 검색 시작 (모든 키워드 앞에 '음식' 추가)
--------------------------------------------------
[1/381] 제육볶음 -> 음식 제육볶음
  블로그: 723,030, 뉴스: 12,006, 카페: 68,712, 웹: 1,713,289
  총합: 2,517,037
[2/381] 매운제육볶음 -> 음식 매운제육볶음
  블로그: 105,336, 뉴스: 1,033, 카페: 8,859, 웹: 542,850
  총합: 658,078
[3/381] 두부제육볶음 -> 음식 두부제육볶음
  블로그: 130,639, 뉴스: 1,092, 카페: 12,785, 웹: 568,800
  총합: 713,316
[4/381] 된장찌개 -> 음식 된장찌개
  블로그: 1,915,540, 뉴스: 29,037, 카페: 125,314, 웹: 2,643,693
  총합: 4,713,584
[5/381] 김치찌개 -> 음식 김치찌개
  블로그: 1,251,767, 뉴스: 42,179, 카페: 143,205, 웹: 4,964,078
  총합: 6,401,229
[6/381] 청국장찌개 -> 음식 청국장찌개
  블로그: 198,028, 뉴스: 3,499, 카페: 19,246, 웹: 886,550
  총합: 1,107,323
[7/381] 콩나물무침 -> 음식 콩나물무침
  블로그: 684,904, 뉴스: 4,241, 카페: 48,485, 웹: 1,208,379
  총합: 1,946,009
[8/381] 시금치나물 -> 음식 시금치나

  블로그: 220,185, 뉴스: 2,929, 카페: 15,224, 웹: 206,572
  총합: 444,910
[88/381] 조개구이 -> 음식 조개구이
  블로그: 598,143, 뉴스: 8,646, 카페: 41,066, 웹: 1,464,282
  총합: 2,112,137
[89/381] 키조개 -> 음식 키조개
  블로그: 243,795, 뉴스: 4,591, 카페: 14,144, 웹: 366,718
  총합: 629,248
[90/381] 가리비 -> 음식 가리비
  블로그: 844,268, 뉴스: 9,798, 카페: 50,954, 웹: 922,469
  총합: 1,827,489
[91/381] 파전 -> 음식 파전
  블로그: 691,919, 뉴스: 12,554, 카페: 57,328, 웹: 1,938,119
  총합: 2,699,920
[92/381] 김치전 -> 음식 김치전
  블로그: 379,470, 뉴스: 6,457, 카페: 33,278, 웹: 11,501,876
  총합: 11,921,081
[93/381] 해물전 -> 음식 해물전
  블로그: 27,587, 뉴스: 746, 카페: 3,013, 웹: 2,861,472
  총합: 2,892,818
[94/381] 골뱅이무침 -> 음식 골뱅이무침
  블로그: 138,292, 뉴스: 1,684, 카페: 16,011, 웹: 446,814
  총합: 602,801
[95/381] 오징어무침 -> 음식 오징어무침
  블로그: 433,724, 뉴스: 4,792, 카페: 45,077, 웹: 1,299,435
  총합: 1,783,028
[96/381] 멍게무침 -> 음식 멍게무침
  블로그: 116,097, 뉴스: 1,184, 카페: 5,335, 웹: 313,457
  총합: 436,073
[97/381] 곱창 -> 음식 곱창
  블로그: 1,513,855, 뉴스: 22,507, 카페: 126,602, 웹: 3,392,078
  총합: 5,055,042
[98/381] 막창 -> 음식 막창
  블로그: 61

[180/381] 필라델피아롤 -> 음식 필라델피아롤
  블로그: 3,936, 뉴스: 93, 카페: 422, 웹: 253,414
  총합: 257,865
[181/381] 돈코츠라멘 -> 음식 돈코츠라멘
  블로그: 180,469, 뉴스: 1,290, 카페: 7,436, 웹: 312,587
  총합: 501,782
[182/381] 차슈라멘 -> 음식 차슈라멘
  블로그: 155,984, 뉴스: 542, 카페: 4,111, 웹: 70,242
  총합: 230,879
[183/381] 미소라멘 -> 음식 미소라멘
  블로그: 45,899, 뉴스: 673, 카페: 2,913, 웹: 392,242
  총합: 441,727
[184/381] 쇼유라멘 -> 음식 쇼유라멘
  블로그: 24,308, 뉴스: 240, 카페: 1,440, 웹: 56,954
  총합: 82,942
[185/381] 시오라멘 -> 음식 시오라멘
  블로그: 31,633, 뉴스: 249, 카페: 1,509, 웹: 115,194
  총합: 148,585
[186/381] 맑은라멘 -> 음식 맑은라멘
  블로그: 32,760, 뉴스: 298, 카페: 1,620, 웹: 392,582
  총합: 427,260
[187/381] 매운라멘 -> 음식 매운라멘
  블로그: 174,481, 뉴스: 975, 카페: 6,664, 웹: 464,971
  총합: 647,091
[188/381] 마제소바 -> 음식 마제소바
  블로그: 210,008, 뉴스: 516, 카페: 3,215, 웹: 293,832
  총합: 507,571
[189/381] 매운미소 -> 음식 매운미소
  블로그: 130,856, 뉴스: 3,469, 카페: 11,847, 웹: 763,428
  총합: 909,600
[190/381] 츠케멘 -> 음식 츠케멘
  블로그: 47,129, 뉴스: 207, 카페: 2,329, 웹: 81,021
  총합: 130,686
[191/381] 아부라소바 -> 음식 아부라소바
  블로그: 28,746, 뉴스: 1

[273/381] 월남쌈 -> 음식 월남쌈
  블로그: 444,082, 뉴스: 4,175, 카페: 35,658, 웹: 565,153
  총합: 1,049,068
[274/381] 고이쿤 -> 음식 고이쿤
  블로그: 9, 뉴스: 0, 카페: 2, 웹: 15
  총합: 26
[275/381] 반미 -> 음식 반미
  블로그: 227,745, 뉴스: 3,603, 카페: 40,262, 웹: 692,091
  총합: 963,701
[276/381] 바게트 -> 음식 바게트
  블로그: 778,073, 뉴스: 7,784, 카페: 49,458, 웹: 1,664,696
  총합: 2,500,011
[277/381] 팟타이 -> 음식 팟타이
  블로그: 481,353, 뉴스: 4,043, 카페: 33,084, 웹: 222,108
  총합: 740,588
[278/381] 태국볶음면 -> 음식 태국볶음면
  블로그: 82,659, 뉴스: 1,612, 카페: 4,419, 웹: 786,263
  총합: 874,953
[279/381] 그린커리 -> 음식 그린커리
  블로그: 77,099, 뉴스: 1,002, 카페: 5,347, 웹: 363,172
  총합: 446,620
[280/381] 레드커리 -> 음식 레드커리
  블로그: 46,449, 뉴스: 913, 카페: 2,937, 웹: 317,356
  총합: 367,655
[281/381] 팬낭커리 -> 음식 팬낭커리
  블로그: 2, 뉴스: 0, 카페: 0, 웹: 0
  총합: 2
[282/381] 똠얌꿍 -> 음식 똠얌꿍
  블로그: 78,912, 뉴스: 739, 카페: 6,270, 웹: 79,078
  총합: 164,999
[283/381] 똠얌갈비 -> 음식 똠얌갈비
  블로그: 3,556, 뉴스: 33, 카페: 195, 웹: 4,058
  총합: 7,842
[284/381] 새콤매운국물 -> 음식 새콤매운국물
  블로그: 34,021, 뉴스: 211, 카페: 1,350, 웹: 81,178
  총합: 116,760
[285

[365/381] 일식파스타 -> 음식 일식파스타
  블로그: 149,841, 뉴스: 2,804, 카페: 8,758, 웹: 995,575
  총합: 1,156,978
[366/381] 우동파스타 -> 음식 우동파스타
  블로그: 249,301, 뉴스: 2,026, 카페: 10,888, 웹: 779,583
  총합: 1,041,798
[367/381] 한식뷔페 -> 음식 한식뷔페
  블로그: 441,388, 뉴스: 11,300, 카페: 45,201, 웹: 2,298,634
  총합: 2,796,523
[368/381] 무한리필 -> 음식 무한리필
  블로그: 1,378,918, 뉴스: 10,461, 카페: 64,508, 웹: 2,381,249
  총합: 3,835,136
[369/381] 샐러드바 -> 음식 샐러드바
  블로그: 1,470,271, 뉴스: 15,419, 카페: 64,175, 웹: 2,381,263
  총합: 3,931,128
[370/381] 샐러드뷔페 -> 음식 샐러드뷔페
  블로그: 660,393, 뉴스: 9,085, 카페: 37,916, 웹: 997,791
  총합: 1,705,185
[371/381] 호텔뷔페 -> 음식 호텔뷔페
  블로그: 703,025, 뉴스: 23,823, 카페: 138,688, 웹: 3,624,765
  총합: 4,490,301
[372/381] 브런치뷔페 -> 음식 브런치뷔페
  블로그: 66,110, 뉴스: 2,217, 카페: 5,278, 웹: 1,118,844
  총합: 1,192,449
[373/381] 비건버거 -> 음식 비건버거
  블로그: 26,945, 뉴스: 1,878, 카페: 1,841, 웹: 362,518
  총합: 393,182
[374/381] 두부스테이크 -> 음식 두부스테이크
  블로그: 166,487, 뉴스: 4,022, 카페: 19,407, 웹: 961,001
  총합: 1,150,917
[375/381] 템페 -> 음식 템페
  블로그: 14,738, 뉴스: 498, 카페: 1,354,